# GlassBox on Gemma-2-2B-it

Interactive mechanistic interpretability for **google/gemma-2-2b-it** — SAE features (Gemma Scope), live feature ablation / steering, causal context-attribution, and the knowledge-conflict hallucination experiment. The whole app runs behind one public URL.

**Before you run:**
1. **Runtime -> Change runtime type -> T4 GPU.**
2. Accept the **gemma-2-2b-it** license with the *same* HF account as your token (Gemma Scope SAEs are public — no license needed):
   - https://huggingface.co/google/gemma-2-2b-it
3. **Runtime -> Run all.** Paste your HF token at the login prompt, then open the URL printed by the launch cell.

First Gemma load (~1-2 min) happens when you pick `gemma-2-2b-it` in the model selector or run the experiment cell. GPT-2 is the default and loads instantly.

> **If Gemma makes the backend go "offline":** a free Colab T4 (~13 GB system RAM) is tight for loading a 2B model. Use a **High-RAM** runtime if you have one, or run the **Kaggle** version `colab/glassbox_kaggle.ipynb` (T4 x2 = ~29 GB RAM) — that loads Gemma comfortably. GPT-2 mode works on any runtime.

In [ ]:
!git clone -b main https://github.com/santoshcheethiralame-dot/GlassBox.git
%cd GlassBox
!pip install -q sae_lens accelerate fastapi "uvicorn[standard]" scikit-learn requests

In [ ]:
from huggingface_hub import login
login()  # paste a token from an account that accepted the two licenses above

In [ ]:
# build the React frontend and hand it to FastAPI to serve
!cd frontend && npm ci --no-audit --no-fund --silent && npm run build
!rm -rf backend/static && cp -r frontend/dist backend/static
print("frontend built ->", __import__("os").listdir("backend/static"))

In [ ]:
# Give the free Colab T4 (~13 GB RAM) extra headroom so the one-time Gemma load
# can't get OOM-killed. Best-effort: if the runtime won't allow swap we just continue.
import subprocess
try:
    subprocess.run(
        "fallocate -l 12G /swapfile && chmod 600 /swapfile && mkswap /swapfile && swapon /swapfile",
        shell=True, check=True, capture_output=True, text=True,
    )
    print("added 12G swap — Gemma has room to load")
except subprocess.CalledProcessError as e:
    print("swap unavailable (continuing; use a High-RAM runtime or Kaggle if Gemma OOMs):",
          (e.stderr or "").strip()[:200])
!free -h

In [ ]:
import os, re, time, subprocess, urllib.request, requests

if not os.path.exists("cloudflared"):
    urllib.request.urlretrieve(
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        "cloudflared",
    )
    os.chmod("cloudflared", 0o755)

backend = subprocess.Popen(
    ["uvicorn", "main:app", "--host", "127.0.0.1", "--port", "7860"], cwd="backend"
)

print("starting backend (loads GPT-2; Gemma loads on first use)...")
for _ in range(90):
    try:
        requests.get("http://127.0.0.1:7860/api/health", timeout=2)
        break
    except Exception:
        time.sleep(2)

tunnel = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)
url = None
for line in iter(tunnel.stdout.readline, ""):
    print(line, end="")
    m = re.search(r"https://[-a-z0-9]+\.trycloudflare\.com", line)
    if m:
        url = m.group(0)
        break

print("\n" + "=" * 64)
print("  GlassBox is LIVE ->", url)
print("  In the masthead, switch the model selector to 'gemma-2-2b-it'.")
print("  (first Gemma load takes ~1-2 min)")
print("=" * 64)

import requests

r = requests.post(
    "http://127.0.0.1:7860/api/experiment",
    json={"model_key": "gemma-2-2b-it", "layer": 20},
    timeout=900,
)

if r.status_code != 200 or not r.text.strip():
    print(f"experiment failed (HTTP {r.status_code}).")
    print(repr(r.text[:600]) if r.text.strip()
          else "empty response — the backend was most likely OOM-killed loading Gemma. "
               "Check the launch-cell output for 'Killed', and make sure the swap cell ran.")
else:
    r = r.json()
    print(f"grounded {r['n_grounded']}/{r['n_total']}  ·  confabulated {r['n_confabulated']}\n")
    for row in r["rows"]:
        print(f"  {row['subject']:30} {row['predicted']:14} {row['label']:13} ld={row['logit_diff']:+.2f}")
    f = r["flip"]
    print(f"\nFLIP: {f['subject']}  ·  ablate f/{f['feature']}  [{f['feature_label']}]")
    print(f"  logit_diff (grounded - parametric): {f['ld_before']:+.2f} -> {f['ld_after']:+.2f}  (shift {f['shift']:+.2f})")

In [ ]:
import requests

r = requests.post(
    "http://127.0.0.1:7860/api/experiment",
    json={"model_key": "gemma-2-2b-it", "layer": 20},
    timeout=900,
).json()

print(f"grounded {r['n_grounded']}/{r['n_total']}  ·  confabulated {r['n_confabulated']}\n")
for row in r["rows"]:
    print(f"  {row['subject']:30} {row['predicted']:14} {row['label']:13} ld={row['logit_diff']:+.2f}")
f = r["flip"]
print(f"\nFLIP: {f['subject']}  ·  ablate f/{f['feature']}  [{f['feature_label']}]")
print(f"  logit_diff (grounded - parametric): {f['ld_before']:+.2f} -> {f['ld_after']:+.2f}  (shift {f['shift']:+.2f})")